# Single meeting evaluation

## Goal
Learn how DeepEval's individual metrics relate to the complete meeting evaluator's scores, findings, and source evidence. This example uses the faithful fictional `meeting_001` fixture; it contains no real Fidelity views or market claims.

The notebook makes real, billable requests to your configured judge endpoint. It first measures both metrics individually, then repeats them through the full evaluator to demonstrate evidence-rich output. Skip the individual-metric section on routine runs to avoid those duplicate requests.

Saved output paths use `<PROJECT_ROOT>` in place of the local project folder.

## Setup
Install the project once from the project root with `.venv/bin/python -m pip install -e '.[dev,notebooks]'`, then start `.venv/bin/jupyter lab notebooks`. Select a kernel using that environment. Keep your API key in the project's existing `.env`; do not paste credentials into notebook cells.

In [1]:
from pathlib import Path
from dotenv import load_dotenv

# Works when Jupyter starts from the project root or the notebooks folder.
ROOT = next(path for path in [Path.cwd(), Path.cwd().parent] if (path / "pyproject.toml").exists())
load_dotenv(ROOT / ".env", override=False)
print("Project:", ROOT)

Project: <PROJECT_ROOT>


## Steps
### 1. Load the source and generated summary
The full transcript is the grounding source. No reference summary or external market data is required.

In [2]:
meeting_id = "meeting_001"
transcript_path = ROOT / "examples/data/transcripts" / f"{meeting_id}.txt"
summary_path = ROOT / "examples/data/summaries" / f"{meeting_id}.md"
transcript = transcript_path.read_text(encoding="utf-8")
summary = summary_path.read_text(encoding="utf-8")
print("Transcript:", transcript_path.name, len(transcript), "characters")
print("Summary:", summary_path.name, len(summary), "characters")
print(summary)

Transcript: meeting_001.txt 2070 characters
Summary: meeting_001.md 1566 characters
FICTIONAL SOFTWARE TEST SCENARIO — all roles, figures and views are invented.

## Executive Briefing
- The scenario's annual inflation reading eased from 3.4% to 3.2%, but the fictional Fidelity presenter described moderation as uneven and offered no rate-cut date forecast.
- Short-term yields fell 10 basis points while longer-term yields were roughly unchanged; the discussed talking point was that a higher yield does not eliminate price risk.

## Sales Insights
- Ask advisors about investment horizon, tolerance for interest-rate volatility, and near-term liquidity needs.
- Use the explicitly discussed talking point that a higher yield does not eliminate price risk. No specific fund, launch, or performance promise was discussed.

## Detailed Market Updates and Fidelity Perspectives
- Inflation: the lower reading was one observation rather than proof of a durable trend.
- Rates: the weekly decline applie

### 2. Initialize the judge
Change `MODEL` if your endpoint uses a different supported model ID. `ChatOpenAI` reads the endpoint and credentials loaded above. Sampling parameters are omitted to avoid assumptions about model support.

In [3]:
from langchain_openai import ChatOpenAI
from meeting_eval import MeetingSummaryEvaluator

MODEL = "gpt-6-luna"
llm = ChatOpenAI(model=MODEL, timeout=120, max_retries=2)
evaluator = MeetingSummaryEvaluator(model=llm, max_concurrency=1, env_file=ROOT / ".env")

Warning: DeepEval is configured for read only environment. Test runs will not be written to disk.

### 3. Run individual DeepEval metrics
This is the underlying DeepEval interface. Faithfulness judges included claims against the transcript; coverage judges which important information survives. Both produce 0–1 scores, with different meanings. The adapter uses emitted rubric scores rather than probability-weighted G-Eval because it does not return log probabilities.

Use `await` in Jupyter: the notebook already has a running event loop.

In [4]:
from deepeval.test_case import LLMTestCase
from meeting_eval.judge import LangChainJudge
from meeting_eval.metrics import create_metrics

test_case = LLMTestCase(
    input="Summarize this Capital Markets meeting for Fidelity Institutional sales professionals using only explicitly discussed information.",
    actual_output=summary,
    retrieval_context=[transcript],
)
individual_metrics = create_metrics(LangChainJudge(llm, max_concurrency=1))
for name, metric in individual_metrics.items():
    await metric.a_measure(test_case)
    print(f"{name}: {metric.score:.2f}")
    print(metric.reason)
    print()

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

faithfulness: 1.00
The score is 1.00 because there are no contradictions; the actual output is fully consistent with the retrieval context.



coverage: 1.00
The summary preserves all consequential themes and qualifications: inflation fell from 3.4% to 3.2% without establishing a durable trend; short-term yields declined 10 basis points while longer-term yields were roughly unchanged; the Fidelity view was uneven moderation, data-dependent policy, and no rate-cut date forecast. It also covers balancing income with duration risk, the warning that higher yield does not eliminate price risk, client horizon, volatility tolerance and liquidity discovery, and the absence of a named-product recommendation. Both meaningful Q&A exchanges are represented; omissions are inconsequential.



### 4. Run the complete evaluator
The complete path repeats the scores above and adds a structured diagnostic review, quotation matching, and review status. This is the call to use in ordinary single-meeting experiments.

In [5]:
result = await evaluator.aevaluate(transcript, summary, meeting_id=meeting_id)
print("Execution:", result.execution_status)
print("Review required:", result.review_required)
for name, metric in result.metrics.items():
    print(name, metric.score, metric.status)
    print(metric.reason)
if result.errors:
    print("Evaluation errors:", result.errors)

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<PROJECT_ROOT>/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Execution: completed
Review required: False
faithfulness 1.0 completed
The score is 1.00 because there are no contradictions; the actual output is fully faithful to the retrieval context.
coverage 0.9 completed
The summary covers the major developments and qualifications: inflation fell from 3.4% to 3.2% without establishing a durable trend, short-term yields declined 10 basis points while longer-term yields were roughly unchanged, and the presenter gave no rate-cut date forecast. It also preserves the income-versus-duration-risk discussion, the explicit higher-yield/price-risk talking point, the horizon and liquidity discovery questions, and both substantive Q&As. A minor shortcoming is that it does not consistently identify who made the market observations and sales comments, though it does not materially distort them.


### 5. Inspect findings and quotations
`verified=True` means the exact quoted passage was uniquely located in the selected source. Missing or repeated quotations remain unverified without assigned source lines. It does not prove that the judge's interpretation is correct. An unsupported claim may have a summary quotation but no supporting transcript passage. Uncertain findings should remain uncertain until a reviewer resolves them.

In [6]:
for finding in result.findings:
    print(f"[{finding.severity}] {finding.kind} — {finding.section or 'whole summary'}")
    print(finding.description)
    for evidence in finding.evidence:
        location = f"line {evidence.line_start}" if evidence.line_start else "unmatched quote"
        print(f"  {evidence.source}, {location}, verified={evidence.verified}")
        print(f"  {evidence.quote}")
    if finding.suggested_prompt_change:
        print("Prompt suggestion:", finding.suggested_prompt_change)
    print()
if not result.findings:
    print("No findings were returned. This is not a guarantee that the summary has no errors.")

No findings were returned. This is not a guarantee that the summary has no errors.


## Checks
A successful run should return both core scores. Review status is separate from execution status: a completed evaluation can identify serious summary problems. An API or diagnostic failure is an evaluation limitation, not a low-quality summary.

In [7]:
assert set(result.metrics) == {"faithfulness", "coverage"}
assert all(metric.score is None or 0 <= metric.score <= 1 for metric in result.metrics.values())
print("Review reasons:", result.review_reasons)
print("Run metadata:", result.metadata)

Review reasons: []
Run metadata: {'model': 'gpt-6-luna', 'metrics': ['faithfulness', 'coverage'], 'versions': {'deepeval': '4.3.2', 'langchain-openai': '1.7.3', 'meeting-summary-eval': '0.1.0'}, 'methodology_version': '1', 'sales_policy': 'explicit_discussion_only', 'scoring': 'separate_scores_no_acceptance_gate', 'max_concurrency': 1, 'timestamp': '2026-10-28T13:23:10.667776+00:00', 'diagnostics_status': 'completed', 'transcript_sha256': 'c3e8b83b676ecc2bf453a24683d16188f24ada5c6c65091052691656eb3fc387', 'summary_sha256': '4d91f7447a86d4e233c33c4cd245f242ce983966026b594d671581035f75c896', 'transcript_lines': 15, 'summary_lines': 18, 'qa_present': True}


## Next steps
Try changing `meeting_id` to `meeting_002` and rerun from the data-loading cell. It intentionally loses the source's central concentration and liquidity themes: coverage should expose that failure even though the remaining claims are mostly faithful. `meeting_003` deliberately changes figures, reverses the stated view, and invents a product recommendation; inspect its material-risk findings rather than relying on a score alone.

No automatic pass/fail threshold or weighted average is used. Plausible sales implications are unsupported when the meeting did not explicitly discuss them. See `docs/methodology.md` for rubrics and limitations. The batch notebook compares all three fixtures in one run.